# Notebook 22 — Seven-septile target, class-balanced nested evaluation

**New experimental target:** qcut(q=7), ascending labels `[2,2,0,0,1,1,1]` (2=down, 0=middle, 1=up). Classifier `class_weight="balanced"`.

**Caution:** Class 1 includes the top three septiles (~42.9% of observations), so always compare to the majority-class baseline. Class 0 is a middle-return bucket, not necessarily a zero return.

This is a *new target version*; historical 38–40% results are not directly comparable. Three outer chronological folds, two inner folds. Existing saved models and prospective ledger remain unchanged.


In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
if not (ROOT/'src').exists(): raise RuntimeError('Run from repository root or notebooks directory')
if str(ROOT/'src') not in sys.path: sys.path.insert(0,str(ROOT/'src'))
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.nested_accuracy import run_nested_accuracy,CANDIDATES


In [ ]:
# Requested: three outer folds. Two inner folds for training-only candidate selection.
N_OUTER=3
N_INNER=2
# Start with smaller fit caps. Increase only after validating the full run.
MAX_TRAIN_ROWS=25000
TREE_TRAIN_ROWS=15000
OUTPUT=ROOT/'artifacts'/'nested_accuracy'/'1d'
required=('master_labeled.parquet','latest_inference.parquet','transformers.joblib','manifest.json')
locations=[ROOT/'artifacts'/'master_tables'/'1d',ROOT/'notebooks'/'artifacts'/'master_tables'/'1d']
source=next((p for p in locations if all((p/f).exists() for f in required)),None)
if source is None: raise FileNotFoundError('Notebook 03 master artifacts missing')
master,latest,bundle,manifest=load_master(source)
print('Master:',len(master),'rows,',master.symbol.nunique(),'symbols')
print('Candidates:',[c[0] for c in CANDIDATES])
TARGET_Q=7
BALANCE_CLASSES=True
print("Target: 7 bins -> [2,2,0,0,1,1,1]; class weights balanced")


## Run nested chronological selection

Each outer fold selects a candidate using only its inner training-period folds. Outer scores are recorded **after** selection. Do not use outer scores to retune the candidate grid.

In [ ]:
inner,outer,selections,cutoff=run_nested_accuracy(master,n_outer=N_OUTER,n_inner=N_INNER,max_train_rows=MAX_TRAIN_ROWS,tree_train_rows=TREE_TRAIN_ROWS,target_q=TARGET_Q,balance_classes=BALANCE_CLASSES,output_dir=OUTPUT)
print('Development cutoff:',cutoff)
display(selections)
display(outer)


## Compare accuracy, uncertainty, and training costs

With three outer folds, uncertainty is substantial. This is an exploratory development comparison, **not** evidence of a deployable trading edge.

In [ ]:
display(outer[['accuracy','balanced_accuracy','roc_auc','log_loss','fit_seconds']].agg(['mean','std']))
display(inner.groupby('candidate',as_index=False).agg(inner_accuracy=('accuracy','mean'),inner_balanced_accuracy=('balanced_accuracy','mean')).sort_values('inner_accuracy',ascending=False))
print('Artifacts:',OUTPUT)


## Decision gate for Notebook 23

If nested accuracy remains near 40%, the next priority is to revisit the **economic target definition** (fees, spread, minimum move, and abstention), rather than repeatedly tuning on these outer folds. Do not promote the winner to the prospective ledger without a separately versioned, forward-only shadow trial.

**News/NLP remains a separate future feature source:** timestamped article ingestion belongs in `stock_analytics_v0`; FinBERT-derived ticker/session sentiment can later be joined point-in-time here. No news data is used in this notebook.

In [ ]:
display(outer[["outer_fold","winner","accuracy","majority_baseline_accuracy","balanced_accuracy","up_precision","up_recall","up_prediction_rate","actual_up_rate","roc_auc"]])
print("Mean model accuracy:",round(outer.accuracy.mean(),4))
print("Mean majority baseline accuracy:",round(outer.majority_baseline_accuracy.mean(),4))
print("Mean up precision:",round(outer.up_precision.mean(),4))
print("Compare only against this new q=7 target baseline; old q=3 metrics are not comparable.")
